# 03 — Labels extraction (Dryad Snapshot Serengeti)

Labels come from the Dryad Snapshot Serengeti release 
Coordinates are `LocationX` / `LocationY` in EPSG:32736 (UTM 36S), per ADR-0001.

In [40]:
import pandas as pd

consensus = pd.read_csv("data/consensus_data.csv", parse_dates=["DateTime"])
effort = pd.read_csv("data/search_effort.csv", parse_dates=["Start date", "End date"])

print(consensus.describe())
print(consensus["Babies"])

print(effort.head())


effort = effort.rename(columns={"Site ID": "SiteID", "Start date": "StartDate", "End date": "EndDate"})


           NumImages                    DateTime      LocationX     LocationY  \
count  334671.000000                      334671  334671.000000  3.346710e+05   
mean        2.662893  2012-01-20 14:42:17.248755  712640.019216  9.722322e+06   
min         1.000000         2010-07-16 10:31:20  694892.000000  9.708102e+06   
25%         3.000000         2011-06-12 13:17:33  701611.000000  9.715888e+06   
50%         3.000000         2012-02-22 11:21:47  710085.000000  9.722727e+06   
75%         3.000000         2012-07-21 15:06:26  721832.000000  9.729237e+06   
max        12.000000         2013-04-26 11:30:37  739908.000000  9.736056e+06   
std         0.748432                         NaN   13044.205306  7.821208e+03   

          NumSpecies       Standing        Resting         Moving  \
count  334671.000000  334671.000000  334671.000000  334671.000000   
mean        1.088899       0.459957       0.101134       0.418830   
min         1.000000       0.000000       0.000000       0.0000

In [41]:
SITE_POSITION_TOLERANCE_M = 200

site_coords = consensus.groupby(["SiteID", "LocationX", "LocationY"]).size().rename("n").reset_index()

spread = site_coords.groupby("SiteID").agg(dx=("LocationX", lambda s: s.max() - s.min()),
                                           dy=("LocationY", lambda s: s.max() - s.min()))

spread["position_uncertainty_m"] = (spread.dx**2 + spread.dy**2) ** 0.5

too_far = spread[spread.position_uncertainty_m > SITE_POSITION_TOLERANCE_M]

assert too_far.empty, f"sites moved more than {SITE_POSITION_TOLERANCE_M} m: {too_far.index.tolist()}"

# Most-used coordinate per site, plus its uncertainty
sites = (site_coords.sort_values("n", ascending=False)
                    .drop_duplicates("SiteID")[["SiteID", "LocationX", "LocationY"]]
                    .merge(spread[["position_uncertainty_m"]], on="SiteID"))

# Attach coordinates to every running period; fail if any site is missing
effort_xy = effort.merge(sites, on="SiteID", how="left", validate="many_to_one")
missing = effort_xy[effort_xy["LocationX"].isna()]["SiteID"].unique()
assert len(missing) == 0, f"sites in effort with no coordinates: {missing}"

print(f"{len(sites)} sites, {len(effort_xy)} running periods")
effort_xy.head()

225 sites, 1128 running periods


,SiteID,StartDate,EndDate,LocationX,LocationY,position_uncertainty_m
0,B03,2011-12-30,2012-01-09,695075,9731632,0.0
1,B03,2012-02-28,2012-04-18,695075,9731632,0.0
2,B03,2012-08-04,2012-10-04,695075,9731632,0.0
3,B03,2013-02-08,2013-04-08,695075,9731632,0.0
4,B04,2010-07-18,2010-08-24,695238,9729497,0.0


In [42]:
import geopandas as gpd
import plotly.express as px
import numpy as np

# Stored in EPSG:32736 (meters); reprojected to 4326 only for display, per ADR-0001
gsites = gpd.GeoDataFrame(sites, geometry=gpd.points_from_xy(sites.LocationX, sites.LocationY), crs="EPSG:32736")
disp = gsites.to_crs("EPSG:4326")

disp["lat"], disp["lon"] = disp.geometry.y, disp.geometry.x
disp["moved"] = disp.position_uncertainty_m > 0

disp["position"] = pd.cut(
    disp.position_uncertainty_m,
    bins=[-1, 100, 300, np.inf],
    labels=["≤ 100 m", "100–300 m", "> 300 m"],
)

lat_min, lat_max = disp.lat.min(), disp.lat.max()
lon_min, lon_max = disp.lon.min(), disp.lon.max()
center = {"lat": (lat_min + lat_max) / 2, "lon": (lon_min + lon_max) / 2}
span = max(lat_max - lat_min, lon_max - lon_min) * 1.2  # 20% padding around the edges
zoom = float(np.log2(360 / span)) - 1

color="position"
color_discrete_map={"≤ 100 m": "#2f9e44", "100–300 m": "#f08c00", "> 300 m": "#e03131"}
category_orders={"position": ["≤ 100 m", "100–300 m", "> 300 m"]}
hover_data={"position_uncertainty_m": ":.0f", "lat": False, "lon": False, "position": False}
title="Snapshot Serengeti camera sites — position uncertainty"

fig = px.scatter_map(
    disp, 
    lat="lat", 
    lon="lon",
    color=color, 
    color_discrete_map=color_discrete_map,
    category_orders=category_orders,
    hover_name="SiteID", 
    hover_data=hover_data,
    zoom=zoom, 
    center = center,
    height=550, 
    map_style="open-street-map",
    title=title,
)
fig.update_traces(marker_size=9)
fig.show()


## 1. Classification quality

Each capture is labelled by several volunteers. `agreement = NumVotes / NumClassifications` is the share who picked that species.
A low value means the volunteers weren't sure, and that row could create a false sighting. Below we compare a few cut-offs before choosing `MIN_AGREEMENT`.

In [43]:
SPECIES_MAP = {
    "cheetah": "cheetah",
    "lionFemale": "lion",
    "lionMale": "lion",
    "leopard": "leopard",
    "elephant": "elephant",
    "buffalo": "buffalo",
}

AGREEMENT_CANDIDATES = [0.3, 0.5, 0.7]
target = consensus[consensus.Species.isin(SPECIES_MAP.keys())].copy()

target.Species = target.Species.map(SPECIES_MAP)

# Share of volunteers who picked this species for the capture
target["agreement"] = target.NumVotes / target.NumClassifications

# Fraction of rows each candidate threshold would drop, per species
drop = pd.DataFrame({th: (target.agreement < th).groupby(target.Species).mean() for th in AGREEMENT_CANDIDATES})
print(drop.round(3))

fig = px.histogram(
    target,
    x="agreement",
    facet_col="Species",
    facet_col_wrap=3,
    nbins=20,
    histnorm="probability",
    height=500,
    title="Volunteer agreement per target species",
)
fig.show()

            0.3    0.5    0.7
Species                      
buffalo   0.011  0.081  0.246
cheetah   0.015  0.097  0.241
elephant  0.006  0.029  0.093
leopard   0.022  0.105  0.289
lion      0.038  0.122  0.246


## 2. Target species + agreement filter

Keep the 5 species from notebook 02 and merge `lionFemale` / `lionMale` into `lion`.
Lion votes are summed **before** filtering, so a capture split 0.4 female / 0.4 male still counts as a confident lion.
A row is kept when at least half of the volunteers named the species (`MIN_AGREEMENT = 0.5`).
The `agreement` column is kept so a stricter cut-off can be applied later without rebuilding.

In [ ]:
MIN_AGREEMENT = 0.5

SPECIES_MAP = {
    "cheetah": "cheetah",
    "lionFemale": "lion",
    "lionMale": "lion",
    "leopard": "leopard",
    "elephant": "elephant",
    "buffalo": "buffalo",
}

sightings = (target.groupby(["CaptureEventID", "Species"], as_index=False)
                   .agg(SiteID=("SiteID", "first"), DateTime=("DateTime", "first"),
                        NumVotes=("NumVotes", "sum"), NumClassifications=("NumClassifications", "first")))

sightings["agreement"] = (sightings.NumVotes / sightings.NumClassifications).clip(upper=1)

before = sightings.Species.value_counts()
sightings = (sightings[sightings.agreement >= MIN_AGREEMENT]
             .sort_values(["SiteID", "Species", "DateTime"])
             .reset_index(drop=True))

assert sightings.Species.notna().all()
assert not sightings.duplicated(["CaptureEventID", "Species"]).any()

agreement_per_species = sightings.groupby("Species")["agreement"].first()

info = pd.DataFrame(
    {
        "before": before, 
        "kept": sightings.Species.value_counts(), 
        "agreement": agreement_per_species
    }
)

print(info)



          before   kept  agreement
Species                           
buffalo    13672  12569   0.928571
cheetah     1272   1148   0.800000
elephant   10178   9882   0.944444
leopard      228    204   0.846154
lion        4177   3658   1.000000


## 3. Spotted labels

One row per photo of a target species: what was seen, by which camera, where (EPSG:32736) and when.
These are the facts each species model is cross-checked against: for every photo, does the model predict the species at that position and time?

In [56]:
SPECIES = list(dict.fromkeys(SPECIES_MAP.values()))

spotted = (sightings.merge(sites, on="SiteID", how="left", validate="many_to_one")
                    [["CaptureEventID", "Species", "SiteID", "DateTime",
                      "LocationX", "LocationY", "position_uncertainty_m", "agreement"]])
assert spotted.LocationX.notna().all()

print(f"{len(spotted)} spotted labels")
print(spotted.Species.value_counts())

spotted.head()

27461 spotted labels
Species
buffalo     12569
elephant     9882
lion         3658
cheetah      1148
leopard       204
Name: count, dtype: int64


,CaptureEventID,Species,SiteID,DateTime,LocationX,LocationY,position_uncertainty_m,agreement
0,ASG000c5gh,elephant,B03,2012-03-18 12:49:21,695075,9731632,0.0,0.944444
1,ASG000c55j,elephant,B03,2012-03-18 12:50:38,695075,9731632,0.0,0.696970
2,ASG000c5dk,elephant,B03,2012-03-18 13:03:06,695075,9731632,0.0,1.000000
3,ASG000c5he,elephant,B03,2012-03-18 13:05:12,695075,9731632,0.0,1.000000
4,ASG000c5i5,elephant,B03,2012-03-18 13:12:48,695075,9731632,0.0,0.916667


## 4. Map

Number of spotted labels per site. Pick the species in the dropdown.
Grey sites never photographed that species. Each species gets its own colour scale.

In [60]:
import plotly.graph_objects as go

# Spotted labels per site and species -> where only spotted is true ( n > 0 )
per_site = (
    spotted.groupby(["SiteID", "Species"]).size().rename("n").reset_index()
           .merge(disp[["SiteID", "lat", "lon"]], on="SiteID", validate="many_to_one")
)

print(per_site)

    SiteID   Species    n       lat        lon
0      B03  elephant  232 -2.426850  34.754378
1      B04   buffalo   21 -2.446155  34.755868
2      B04  elephant   43 -2.446155  34.755868
3      B05   buffalo   46 -2.467274  34.752785
4      B05  elephant   38 -2.467274  34.752785
..     ...       ...  ...       ...        ...
711    V10  elephant    8 -2.566091  35.155653
712    V10      lion    9 -2.566091  35.155653
713    V11   cheetah    6 -2.590082  35.157671
714    V13   buffalo    2 -2.628957  35.157657
715    V13      lion    6 -2.628957  35.157657

[716 rows x 5 columns]


In [61]:

# Sequential blue: light = few labels, dark = many
BLUES = [
    [0, "#86b6ef"], 
    [0.5, "#2a78d6"], 
    [1, "#0d366b"]
]

hovertemplate = "<b>%{text}</b><br>%{customdata} spotted<extra></extra>"

fig = go.Figure()
for sp in SPECIES:
    data_each_specie = per_site[per_site.Species == sp]

    for seen_any, part in [(False, data_each_specie[data_each_specie.n == 0]), (True, data_each_specie[data_each_specie.n > 0])]:
        marker = (dict(size=10, color=part.n, colorscale=BLUES, cmin=0, cmax=part.n.max(),
                       colorbar=dict(title="spotted", thickness=12)))
        
        fig.add_trace(go.Scattermap(
            lat=part.lat, 
            lon=part.lon, 
            mode="markers", 
            marker=marker,
            name="spotted", 
            text=part.SiteID,
            customdata=part.n, 
            hovertemplate=hovertemplate,
            visible=sp == SPECIES[0],
        ))

# Two traces per species; the dropdown shows one species at a time
buttons = [dict(label=sp, 
                method="update",
                args=[{"visible": [s == sp for s in SPECIES for _ in range(2)]},
                      {"title": f"{sp}: spotted labels per site"}])
           for sp in SPECIES]

fig.update_layout(
    map=dict(style="carto-positron", center=center, zoom=zoom),
    updatemenus=[dict(buttons=buttons, x=0, y=1.08, xanchor="left")],
    title=f"{SPECIES[0]}: spotted labels per site",
    legend=dict(x=0.99, xanchor="right", y=0.01),
    height=600,
    margin=dict(t=80, l=0, r=0, b=0),
)

fig.show()